# AI Immune System Challenge - Quick Baseline (No CV)
Single training run with DeBERTa-v3-base on Colab GPU, then create submission.csv.

## セル1: 必要ライブラリのインストール
`transformers`と`scikit-learn`をインストール/更新します。Colab実行時に最初に1回だけ必要です。

In [ ]:
!pip -q install -U transformers scikit-learn

## セル2: Google Driveのマウント
ColabからDriveをマウントし、データをDrive上のパス指定で読み込めるようにします。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## セル3: 基本設定とGPU確認
乱数シードを固定し、`torch`のバージョンとGPU利用可否を確認します。

In [ ]:
import os
import csv
import json
import random
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Reproducibility settings
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True)
except Exception as e:
    print('deterministic warning:', e)

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## セル4: パスと学習ハイパーパラメータ
入力ファイル/出力ファイルのパス、モデル名、学習率、早期終了設定などをまとめて設定します。

In [ ]:
# Base directory on Google Drive (edit this)
DATA_DIR = '/content/drive/MyDrive/AI-Immune-System-Challenge'

# Input/output paths
TRAIN_PATH = os.path.join(DATA_DIR, 'train_labeled_comp.jsonl')
TEST_PATH = os.path.join(DATA_DIR, 'test_labeled_comp.jsonl')
SOLUTION_FORMAT_PATH = os.path.join(DATA_DIR, 'solution_format.csv')
OUTPUT_PATH = os.path.join(DATA_DIR, 'submission.csv')

# Model / optimization
# MODEL_NAME = 'microsoft/deberta-v3-base'
# MODEL_NAME = 'Qwen/Qwen3-Embedding-8B'
MODEL_NAME = 'FacebookAI/roberta-base'
MAX_LEN = 512
LR = 2e-5
TRAIN_BS = 32
EVAL_BS = 32

# Early stopping settings
# MAX_EPOCHS: upper bound only (training can stop earlier)
# EARLY_STOPPING_PATIENCE: stop if validation accuracy does not improve for this many eval rounds
# EARLY_STOPPING_THRESHOLD: minimum delta to count as an improvement
MAX_EPOCHS = 1000
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_THRESHOLD = 0.0

print('TRAIN_PATH:', TRAIN_PATH)
print('TEST_PATH:', TEST_PATH)
print('SOLUTION_FORMAT_PATH:', SOLUTION_FORMAT_PATH)
print('OUTPUT_PATH:', OUTPUT_PATH)


## セル5: データ読み込み
`jsonl`を読み込み、ラベルを`FALSE=0`,`TRUE=1`へ変換します。件数とクラス内訳も表示します。

In [ ]:
LABEL_TO_ID = {'FALSE': 0, 'TRUE': 1}
ID_TO_LABEL = {0: 'FALSE', 1: 'TRUE'}

def load_jsonl(path, has_label=True):
    texts, labels = [], []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            obj = json.loads(line)
            texts.append(obj['text'])
            if has_label:
                labels.append(LABEL_TO_ID[obj['label']])
    if has_label:
        return texts, np.array(labels, dtype=np.int64)
    return texts

train_texts, train_labels = load_jsonl(TRAIN_PATH, has_label=True)
test_texts = load_jsonl(TEST_PATH, has_label=False)

print('train size:', len(train_texts))
print('test size:', len(test_texts))
print('train positives:', int(train_labels.sum()), 'negatives:', int((train_labels==0).sum()))

## セル6: train/valid分割
学習データを層化分割して、検証用データを作成します（CVは使わない単発検証）。

In [ ]:
tr_texts, va_texts, tr_labels, va_labels = train_test_split(
    train_texts, train_labels,
    test_size=0.15,
    random_state=SEED,
    stratify=train_labels
)

print('train split:', len(tr_texts), 'valid split:', len(va_texts))

## セル7: トークナイズとDataset作成
テキストをトークナイズし、`Trainer`が扱えるPyTorch Datasetを作成します。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=MAX_LEN
        )
        self.labels = labels

    def __len__(self):
        return len(self.enc['input_ids'])

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = TextDataset(tr_texts, tr_labels)
valid_ds = TextDataset(va_texts, va_labels)
test_ds = TextDataset(test_texts, None)

## 追加セル: ミニ学習テスト（デバッグ）
本学習の前に32件だけで短時間学習し、`loss`が有限値で推移するか確認します。ここで異常なら本学習に進まないで設定/依存関係を見直します。

In [ ]:
# Quick debug run on a tiny subset (stability-focused)
small_n = 32
small_texts = tr_texts[:small_n]
small_labels = tr_labels[:small_n]

print('debug subset size:', len(small_texts), 'pos:', int(np.sum(small_labels)), 'neg:', int(np.sum(small_labels==0)))

small_ds = TextDataset(small_texts, small_labels)

debug_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    problem_type='single_label_classification'
)
debug_model.gradient_checkpointing_enable()

debug_args = TrainingArguments(
    output_dir='outputs_debug',
    num_train_epochs=1,
    learning_rate=5e-6,
    warmup_ratio=0.1,
    weight_decay=0.01,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=1,
    eval_strategy='no',
    save_strategy='no',
    logging_steps=1,
    logging_nan_inf_filter=False,
    fp16=False,
    bf16=False,
    report_to='none',
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=0,
    max_grad_norm=0.5
)

debug_trainer = Trainer(
    model=debug_model,
    args=debug_args,
    train_dataset=small_ds
)

debug_train_out = debug_trainer.train()
print('debug train finished:', debug_train_out)
print('If loss is finite and not NaN/Inf, proceed to full training cell.')


## セル8: モデル学習（Early Stoppingあり）
`DeBERTa-v3-base`を読み込み、`Trainer`で学習します。各epochで検証し、`val accuracy`が一定回数改善しなければ学習を停止します。

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    return {'accuracy': accuracy_score(labels, preds)}

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

args = TrainingArguments(
    output_dir='outputs_baseline',
    num_train_epochs=MAX_EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    logging_steps=50,
    fp16=False,
    bf16=False,
    report_to='none',
    seed=SEED,
    data_seed=SEED,
    full_determinism=True,
    dataloader_num_workers=0
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD
        )
    ]
)

trainer.train()
eval_metrics = trainer.evaluate()
print(eval_metrics)

## セル9: テスト推論と提出ファイル作成
テストデータを推論し、最適閾値で`TRUE/FALSE`へ変換して`submission.csv`を保存します。

In [ ]:
test_out = trainer.predict(test_ds)
test_probs = torch.softmax(torch.tensor(test_out.predictions), dim=1)[:, 1].numpy()
test_pred = (test_probs >= 0.5).astype(int)

# Keep row count aligned with solution format
with open(SOLUTION_FORMAT_PATH, newline='', encoding='utf-8') as f:
    n_rows = sum(1 for _ in csv.reader(f)) - 1

assert len(test_pred) == n_rows, f'mismatch: pred={len(test_pred)} format={n_rows}'

with open(OUTPUT_PATH, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['label'])
    for p in test_pred:
        w.writerow([ID_TO_LABEL[int(p)]])

print('saved:', OUTPUT_PATH, 'rows:', len(test_pred))